In [6]:
pip install mne

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install "defusedxml<=0.7.1"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import mne
from mne.datasets.eyelink import data_path
from mne.preprocessing.eyetracking import read_eyelink_calibration
from mne.viz.eyetracking import plot_gaze

et_fpath = data_path() / "eeg-et" / "sub-01_task-plr_eyetrack.asc"
eeg_fpath = data_path() / "eeg-et" / "sub-01_task-plr_eeg.mff"

raw_et = mne.io.read_raw_eyelink(et_fpath, create_annotations=["blinks"])
raw_eeg = mne.io.read_raw_egi(eeg_fpath, events_as_annotations=True).load_data()
raw_eeg.filter(1, 30)

Loading C:\Users\pc\mne_data\MNE-eyelink-data\eeg-et\sub-01_task-plr_eyetrack.asc
Pixel coordinate data detected.Pass `scalings=dict(eyegaze=1e3)` when using plot method to make traces more legible.
Pupil-size area detected.
Found 64 button event(s) in this file.
No button events found in this file.
There are 2 recording blocks in this file. Times between blocks will be annotated with BAD_ACQ_SKIP.
Reading EGI MFF Header from C:\Users\pc\mne_data\MNE-eyelink-data\eeg-et\sub-01_task-plr_eeg.mff...
    Reading events ...
    Assembling measurement info ...
    Excluding events {} ...
Reading 0 ... 190020  =      0.000 ...   190.020 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower tra

<RawMff | signal1.bin, 134 x 190021 (190.0 s), ~194.4 MiB, data loaded>

In [3]:
raw_et.info

<Info | 7 non-empty values
 bads: []
 ch_names: xpos_right, ypos_right, pupil_right, DIN, x_head, y_head, distance
 chs: 2 Eye-tracking (Gaze position), 1 Eye-tracking (Pupil size), 1 Stimulus, 3 misc
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 500.0 Hz
 meas_date: 2023-06-29 20:07:21 UTC
 nchan: 7
 projs: []
 sfreq: 1000.0 Hz
>

In [5]:
from mne.datasets.eyelink import data_path
import os

# --- Look at the Eyelink ASC file (text file) ---
print("=== Eyelink ASC file preview ===")
with open(et_fpath, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(100):   # print first 20 lines
        print(f.readline().strip())

# --- Look at the EEG MFF folder structure ---
print("\n=== EEG MFF folder contents ===")
for root, dirs, files in os.walk(eeg_fpath):
    level = root.replace(str(eeg_fpath), "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for fname in files:
        print(f"{subindent}{fname}")


=== Eyelink ASC file preview ===
** CONVERTED FROM C:\Users\Display\Desktop\ACAR_Q1K_protocol_2022_11_25_deploy\7_Q1K_ACAR_PLR_pfp_v1_01_deploy\results\scott\scott.edf using edfapi 3.1 Win32 Nov 21 2017 on Fri Jun 30 03:26:24 2023
** DATE: Thu Jun 29 20:07:21 2023
** TYPE: EDF_FILE BINARY EVENT SAMPLE TAGGED
** VERSION: EYELINK II 1
** SOURCE: EYELINK CL
** EYELINK II CL v5.15 Jan 24 2018
** CAMERA: Eyelink GL Version 1.2 Sensor=AG7
** SERIAL NUMBER: CLG-BFE01
** CAMERA_CONFIG: BFE01200.SCD
** RECORDED BY Q1K_ACAR_PLR_pfp_v1_01
** SREB2.2.299 WIN32 LID:96F1E67 Mod:2023.04.17 19:56 EDT
**

MSG	14660046 !CMD 0 select_parser_configuration 0
MSG	14660263 !CMD 0 fixation_update_interval = 50
MSG	14660263 !CMD 0 fixation_update_accumulate = 50
MSG	14660268 !CMD 1 auto_calibration_messages = YES
MSG	14660336 ;NTPT:188FCCBFBA8;SMSG:__NTP_CLOCK_SYNC__;DIFF:INITIAL;TRAC:14660336;REFT:30609;ESNT:188FCCBFBA8
MSG	14666328 DISPLAY_COORDS 0 0 1919 1079
MSG	14666328 RETRACE_INTERVAL  16.651038269
MSG	

In [4]:
import pandas as pd

et_fpath = data_path() / "eeg-et" / "sub-01_task-plr_eyetrack.asc"

samples = []
with open(et_fpath, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        parts = line.strip().split()
        # Sample lines start with a timestamp (integer) not with MSG/INPUT/etc.
        if parts and parts[0].isdigit():
            timestamp = int(parts[0])
            try:
                x = float(parts[1])   # gaze x
                y = float(parts[2])   # gaze y
                pupil = float(parts[3])  # pupil size (varies by config)
                samples.append((timestamp, x, y, pupil))
            except ValueError:
                continue

df = pd.DataFrame(samples, columns=["time", "x", "y", "pupil"])
print(df.head())
print(df.shape)


       time      x      y   pupil
0  14720720  983.6  492.8  1864.0
1  14720721  983.8  492.6  1863.0
2  14720722  984.0  492.5  1862.0
3  14720723  984.5  492.7  1862.0
4  14720724  985.0  492.6  1863.0
(136744, 4)
